In [5]:
from ESRNN.m4_data import prepare_m4_data
from ESRNN.utils_evaluation import evaluate_prediction_owa

from ESRNN import ESRNN


In [6]:
import numpy; numpy.__version__

'1.16.6'

In [7]:
# mkdir ./data if not exists!

X_train_df, y_train_df, X_test_df, y_test_df = prepare_m4_data(
    dataset_name="Yearly", directory="./data", num_obs=1000
)

In [8]:
X_train_df

,unique_id,ds,x
0,Y1,1970-01-01,Macro
1,Y1,1970-01-02,Macro
2,Y1,1970-01-03,Macro
3,Y1,1970-01-04,Macro
4,Y1,1970-01-05,Macro
...,...,...,...
34876,Y999,1970-01-16,Macro
34877,Y999,1970-01-17,Macro
34878,Y999,1970-01-18,Macro
34879,Y999,1970-01-19,Macro


In [9]:
X_train_df.x.unique()

array(['Macro'], dtype=object)

In [10]:
X_train_df.ds

0       1970-01-01
1       1970-01-02
2       1970-01-03
3       1970-01-04
4       1970-01-05
           ...    
34876   1970-01-16
34877   1970-01-17
34878   1970-01-18
34879   1970-01-19
34880   1970-01-20
Name: ds, Length: 34881, dtype: datetime64[ns]

In [11]:
y_train_df

,unique_id,ds,y
0,Y1,1970-01-01,5172.1
1,Y1,1970-01-02,5133.5
2,Y1,1970-01-03,5186.9
3,Y1,1970-01-04,5084.6
4,Y1,1970-01-05,5182.0
...,...,...,...
34876,Y999,1970-01-16,1742.0
34877,Y999,1970-01-17,1420.0
34878,Y999,1970-01-18,971.0
34879,Y999,1970-01-19,563.0


In [12]:
# X_train_df
# y_train_df
# X_test_df
y_test_df

,unique_id,ds,y,y_hat_naive2
0,Y1,1970-02-01,7290.2,7261.1
1,Y1,1970-02-02,7392.6,7261.1
2,Y1,1970-02-03,7527.6,7261.1
3,Y1,1970-02-04,7594.8,7261.1
4,Y1,1970-02-05,7720.7,7261.1
...,...,...,...,...
5995,Y999,1970-01-22,469.0,383.0
5996,Y999,1970-01-23,328.0,383.0
5997,Y999,1970-01-24,483.0,383.0
5998,Y999,1970-01-25,437.0,383.0


In [14]:
y_test_df.unique_id.value_counts()

Y971    6
Y269    6
Y263    6
Y552    6
Y922    6
       ..
Y846    6
Y564    6
Y199    6
Y853    6
Y865    6
Name: unique_id, Length: 1000, dtype: int64

## Train

In [ ]:
# Instantiate model
model = ESRNN(
    max_epochs=25,
    freq_of_test=5,
    batch_size=4,
    learning_rate=1e-4,
    per_series_lr_multip=0.8,
    lr_scheduler_step_size=10,
    lr_decay=0.1,
    gradient_clipping_threshold=50,
    rnn_weight_decay=0.0,
    level_variability_penalty=100,
    testing_percentile=50,
    training_percentile=50,
    ensemble=False,
    max_periods=25,
    seasonality=[],
    input_size=4,
    output_size=6,
    cell_type="LSTM",
    state_hsize=40,
    dilations=[[1], [6]],
    add_nl_layer=False,
    random_seed=1,
    device="cuda", # FIXED ERTI
)

In [ ]:
import torch

# torch.cuda.set_device(0)

torch.set_default_device("cuda") # FIXED ERTI (maybe not needed with the above device)

In [14]:
# Fit model
# If y_test_df is provided the model
# will evaluate predictions on
# this set every freq_test epochs
model.fit(X_train_df, y_train_df, X_test_df, y_test_df)

# Predict on test set
y_hat_df = model.predict(X_test_df)

# Evaluate predictions
final_owa, final_mase, final_smape = evaluate_prediction_owa(
    y_hat_df, y_train_df, X_test_df, y_test_df, naive2_seasonality=1
)

Infered frequency: D
=============== Training ESRNN  ===============

========= Epoch 0 finished =========
Training time: 5.31729
Training loss (50 prc): 1.09051
Testing loss  (50 prc): 0.06279
OWA: 0.982 
SMAPE: 13.947 
MASE: 3.849 
========= Epoch 1 finished =========
Training time: 5.33849
Training loss (50 prc): 1.07978
========= Epoch 2 finished =========
Training time: 5.48857
Training loss (50 prc): 1.04227
========= Epoch 3 finished =========
Training time: 5.77236
Training loss (50 prc): 1.05084
========= Epoch 4 finished =========
Training time: 5.51863
Training loss (50 prc): 1.06271


KeyboardInterrupt: 